# Phase 08 - LangGraph 企业级：02 - 条件边（Conditional Edges）

## 学习目标

1. 掌握 `add_conditional_edges()` 的各种用法
2. 实现多分支路由（3+ 分支）
3. 实现带循环的图（重试模式）
4. 实现多出口条件的图
5. 可视化复杂图结构

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import operator

print("✅ 导入成功")

## 1. 多分支路由（3+ 分支）

一个路由函数返回 Literal 类型，明确所有可能的分支。

In [ ]:
class RouterState(TypedDict):
    query: str
    intent: str
    answer: str


def node_classify(state: RouterState) -> dict:
    """意图分类节点。"""
    query = state['query'].lower()
    
    if any(w in query for w in ['搜索', '查询', 'search', 'find']):
        intent = 'search'
    elif any(w in query for w in ['计算', '算', 'calculate']):
        intent = 'calculate'
    elif any(w in query for w in ['翻译', 'translate']):
        intent = 'translate'
    elif any(w in query for w in ['代码', 'code', '编程']):
        intent = 'code'
    else:
        intent = 'chat'
    
    print(f"  [classify] 查询分类为: {intent}")
    return {"intent": intent}


def node_search(state: RouterState) -> dict:
    print(f"  [search] 执行搜索")
    return {"answer": f"搜索结果: 关于 '{state['query'][:50]}' 的信息..."}


def node_calculate(state: RouterState) -> dict:
    print(f"  [calculate] 执行计算")
    return {"answer": f"计算结果: {state['query'][:50]} = 42 (示例)"}


def node_translate(state: RouterState) -> dict:
    print(f"  [translate] 执行翻译")
    return {"answer": f"翻译结果: '{state['query'][:30]}...' → English version"}


def node_code(state: RouterState) -> dict:
    print(f"  [code] 生成代码")
    return {"answer": f"代码生成: ```python\n# {state['query'][:50]}\nprint('Hello')\n```"}


def node_chat(state: RouterState) -> dict:
    print(f"  [chat] 闲聊模式")
    return {"answer": f"关于'{state['query'][:50]}'，这是一个很好的问题..."}


# 路由函数：返回 Literal 类型便于类型检查
def route_by_intent(state: RouterState) -> Literal["search", "calculate", "translate", "code", "chat"]:
    return state['intent']


# 构建图
builder = StateGraph(RouterState)
builder.add_node("classify", node_classify)
builder.add_node("search", node_search)
builder.add_node("calculate", node_calculate)
builder.add_node("translate", node_translate)
builder.add_node("code", node_code)
builder.add_node("chat", node_chat)

builder.set_entry_point("classify")

# 5个条件分支
builder.add_conditional_edges(
    "classify",
    route_by_intent,
    {
        "search": "search",
        "calculate": "calculate",
        "translate": "translate",
        "code": "code",
        "chat": "chat",
    }
)

# 所有分支最终都到 END
for node_name in ["search", "calculate", "translate", "code", "chat"]:
    builder.add_edge(node_name, END)

router_graph = builder.compile()

print("✅ 5路分支路由器已构建")
print(router_graph.get_graph().draw_mermaid())

In [ ]:
# 测试不同的查询
test_queries = [
    "搜索2025年AI趋势",
    "计算 100 * (1 + 0.05)^10",
    "把这段话翻译成英文",
    "写一个Python快速排序代码",
    "今天天气真好",
]

for query in test_queries:
    print(f"\n{'─' * 50}")
    print(f"查询: {query}")
    result = router_graph.invoke({"query": query, "intent": "", "answer": ""})
    print(f"  意图: {result['intent']}")
    print(f"  回答: {result['answer'][:80]}...")

## 2. 循环与重试（Looping & Retry）

这是 LangGraph 最强大的特性之一：节点可以循环回到自身或其他节点。

In [ ]:
class RetryLoopState(TypedDict):
    task: str
    result: str
    retry_count: int
    max_retries: int
    quality_score: float


def node_generate(state: RetryLoopState) -> dict:
    """生成答案。"""
    retry = state.get('retry_count', 0) + 1
    print(f"  [generate] 第 {retry} 次生成...")
    
    # 模拟：随着重试，质量逐步提高
    quality = 0.5 + retry * 0.15  # 0.65, 0.80, 0.95...
    
    return {
        "retry_count": retry,
        "result": f"[第{retry}次生成] 回答关于 '{state['task'][:30]}' 的内容 (质量: {quality:.2f})",
        "quality_score": min(1.0, quality),
    }


def node_validate(state: RetryLoopState) -> dict:
    """验证答案质量。"""
    score = state['quality_score']
    print(f"  [validate] 质量评分: {score:.2f}")
    return {}  # 不做修改，仅用于日志


def node_finalize(state: RetryLoopState) -> dict:
    """最终化。"""
    print(f"  [finalize] 答案已确认")
    return {"result": f"✅ 已确认: {state['result']}"}


def node_fallback(state: RetryLoopState) -> dict:
    """降级策略。"""
    print(f"  [fallback] 达到最大重试次数，使用保守策略")
    return {"result": f"⚠️ 降级回答: 抱歉，无法生成高质量回答。请尝试简化问题。"}


# 路由函数
def route_after_validate(state: RetryLoopState) -> Literal["generate", "finalize", "fallback"]:
    """根据质量评分决定下一步。"""
    if state['quality_score'] >= 0.85:
        return "finalize"  # 质量达标
    if state['retry_count'] >= state.get('max_retries', 3):
        return "fallback"  # 重试耗尽
    return "generate"  # 重试（循环回去）


# 构建循环图
builder2 = StateGraph(RetryLoopState)
builder2.add_node("generate", node_generate)
builder2.add_node("validate", node_validate)
builder2.add_node("finalize", node_finalize)
builder2.add_node("fallback", node_fallback)

builder2.set_entry_point("generate")
builder2.add_edge("generate", "validate")

# 关键：validate 的条件路由可以指向 generate（形成循环）
builder2.add_conditional_edges(
    "validate",
    route_after_validate,
    {
        "generate": "generate",  # ← 循环！
        "finalize": "finalize",
        "fallback": "fallback",
    }
)

builder2.add_edge("finalize", END)
builder2.add_edge("fallback", END)

retry_graph = builder2.compile()

print("✅ 带验证循环的图已构建")
print(retry_graph.get_graph().draw_mermaid())

In [ ]:
# 执行重试循环图
print("🚀 执行重试验证循环...\n")

result = retry_graph.invoke({
    "task": "复杂的数学证明问题",
    "result": "",
    "retry_count": 0,
    "max_retries": 3,
    "quality_score": 0.0,
})

print(f"\n📊 执行结果:")
print(f"  重试次数: {result['retry_count']}")
print(f"  最终质量: {result['quality_score']:.2f}")
print(f"  最终结果: {result['result']}")

## 3. 多出口条件（Multiple Exit Conditions）

一个节点可以有多个出口条件，每个指向不同的目标。

In [ ]:
class MultiExitState(TypedDict):
    query: str
    answer: str
    confidence: float
    topic_sensitivity: str  # "safe" | "sensitive" | "dangerous"


def node_analyze(state: MultiExitState) -> dict:
    """分析查询。"""
    query = state['query'].lower()
    
    # 检测敏感词
    dangerous = ['hack', '攻击', '破解', '病毒', 'malware']
    sensitive = ['personal', '个人信息', 'password', '密码']
    
    if any(w in query for w in dangerous):
        sensitivity = 'dangerous'
    elif any(w in query for w in sensitive):
        sensitivity = 'sensitive'
    else:
        sensitivity = 'safe'
    
    print(f"  [analyze] 敏感度: {sensitivity}")
    return {"topic_sensitivity": sensitivity}


def node_respond(state: MultiExitState) -> dict:
    print(f"  [respond] 正常回答")
    return {"answer": f"关于 '{state['query'][:50]}' 的回答..."}


def node_warn_and_respond(state: MultiExitState) -> dict:
    print(f"  [warn] 带警告的回答")
    return {"answer": f"⚠️ 请注意：您的问题涉及敏感信息。回答如下：..."}


def node_reject(state: MultiExitState) -> dict:
    print(f"  [reject] 拒绝回答")
    return {"answer": "🚫 抱歉，此问题违反安全策略，无法回答。"}


def node_human_review(state: MultiExitState) -> dict:
    print(f"  [human_review] 转交人工审核")
    return {"answer": "⏳ 此问题需要人工审核，已加入审核队列。"}


# 路由
def route_by_sensitivity(state: MultiExitState) -> Literal["respond", "warn", "reject"]:
    sensitivity = state['topic_sensitivity']
    if sensitivity == 'safe':
        return "respond"
    elif sensitivity == 'sensitive':
        return "warn"
    else:
        return "reject"


# 构建
builder3 = StateGraph(MultiExitState)
builder3.add_node("analyze", node_analyze)
builder3.add_node("respond", node_respond)
builder3.add_node("warn", node_warn_and_respond)
builder3.add_node("reject", node_reject)
builder3.add_node("human_review", node_human_review)

builder3.set_entry_point("analyze")

builder3.add_conditional_edges(
    "analyze",
    route_by_sensitivity,
    {
        "respond": "respond",
        "warn": "warn",
        "reject": "reject",
    }
)

builder3.add_edge("respond", END)
builder3.add_edge("warn", "human_review")  # 警告后还要人工审核
builder3.add_edge("human_review", END)
builder3.add_edge("reject", END)

multi_exit_graph = builder3.compile()

print("✅ 多出口安全路由图已构建")
print(multi_exit_graph.get_graph().draw_mermaid())

In [ ]:
# 测试三种查询类型
test_cases = [
    ("什么是机器学习？", "安全查询"),
    ("如何保护我的个人信息不被泄露？", "敏感查询"),
    ("教我怎么hack别人的系统", "危险查询"),
]

for query, desc in test_cases:
    print(f"\n{'─' * 50}")
    print(f"{desc}: {query}")
    result = multi_exit_graph.invoke({
        "query": query,
        "answer": "",
        "confidence": 0.0,
        "topic_sensitivity": "",
    })
    print(f"  结果: {result['answer'][:100]}")

## 核心要点总结

1. **条件边语法**：
   ```python
   builder.add_conditional_edges(
       source_node,     # 源节点
       routing_fn,      # 路由函数(state) -> str
       route_map,       # {返回值: 目标节点}
   )
   ```

2. **路由函数**：接收完整State，返回一个字符串（分支名）
3. **循环**：通过将目标节点指回源节点或上游节点实现
4. **多出口**：一个节点可以路由到任意多个目标
5. **终止**：使用 `END` 常量表示图的结束

### 下一步

`03-rag-agent-graph.py` 将把这些概念应用到完整的 RAG Agent 系统中。